# Classification de Spam avec NLP et Régression Logistique

## Objectif
Ce notebook démontre un **pipeline complet de traitement du langage naturel (NLP)** appliqué à la classification de spam.

### Étapes principales :
1. **Chargement et exploration** du dataset
2. **Prétraitement du texte** : Tokenization → Suppression des stopwords → TF-IDF
3. **Encodage des labels** : Conversion des catégories en variables binaires
4. **Classification** : Entraînement avec Régression Logistique
5. **Évaluation** : Métriques adaptées au problème (Accuracy, Precision, Recall, F1-Score)
6. **Optimisation** : Ajustement des poids pour mieux gérer le déséquilibre des classes

## Ressources et Données

**Dataset utilisés :**
- [Spam Emails Dataset](https://www.kaggle.com/datasets/abdallahwagih/spam-emails)


**Modèles et techniques utilisés :**
- **NLP** : Tokenization, Stop Words Removal, TF-IDF Vectorization
- **Classification** : Logistic Regression avec validation croisée
- **Métriques** : Accuracy, Precision, Recall, F1-Score (macro-average)

---

# 1. Configuration et Imports

## Configuration du chemin d'accès

In [3]:
import sys 
sys.path.append(r"E:\cours ifri\Programmation et BD\Python\Pdf et tpcours\Concepts et Application\Apprentissage Automatique\ifri_mini_ml_lib")

## Import des bibliothèques nécessaires

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ifri_mini_ml_lib.preprocessing.preparation.encoding import OneHotEncoder, CategoricalEncoder
from ifri_mini_ml_lib.preprocessing.text.stop_word import StopWordRemover
from ifri_mini_ml_lib.preprocessing.text.tf_idf import TFIDFVectorizer
from ifri_mini_ml_lib.preprocessing.preparation.tokenization import Tokenizer
from ifri_mini_ml_lib.preprocessing.preparation.splitting import DataSplitter

from ifri_mini_ml_lib.metrics.classification import f1_score, recall, precision, accuracy

# Modeles de Classification
from ifri_mini_ml_lib.classification.logistic_regression import LogisticRegression

# Cross Validation
from ifri_mini_ml_lib.model_selection.cross_validation import k_fold_cross_validation


---

# 2. Chargement et Exploration des Données

## Chargement du dataset

In [56]:
df = pd.read_csv("spam.csv")


In [57]:
df.isna().sum()

Category    0
Message     0
dtype: int64

## Vérification des données manquantes

## Séparation Train/Test

No missing values

In [100]:
splitter = DataSplitter(seed=42)


X = df[["Message"]]
y = df["Category"]




X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)

X_train, X_test, y_train, y_test = pd.DataFrame(X_train) , pd.DataFrame(X_test) , pd.DataFrame(y_train), pd.DataFrame(y_test)
# Proportions:
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 4458, Test: 1114


---

# 3. Pipeline NLP (Natural Language Processing)

## Initialisation des outils NLP

Pipeline : **Tokenization → Suppression des Stop Words → TF-IDF Vectorization**

In [98]:
# Pipeline NLP simple: Tokenization → Stopwords → TF-IDF

# 1. Tokenizer
tokenizer = Tokenizer()

# 2. Stop words remover

stopwords_remover = StopWordRemover(language='english')

# 3. TF-IDF

tfidf = TFIDFVectorizer(max_features = 20000)



In [59]:
X.describe()

,Message
count,5572
unique,5157
top,"Sorry, I'll call later"
freq,30


### Étape 1 : **Tokenisation**

In [101]:
X_train_tokens = X_train['Message'].apply(tokenizer.tokenize)
X_train_tokens.head()


4293                                            [g, w, r]
1978    [reply, to, win, 100, weekly, where, will, the...
3989    [hello, sort, of, out, in, town, already, that...
3935    [how, come, guoyang, go, n, tell, her, then, u...
4078    [hey, sathya, till, now, we, dint, meet, not, ...
Name: Message, dtype: object

### Étape 2 : **Suppression des Stop Words**

In [102]:
X_train_without_stop_words = X_train_tokens.apply(stopwords_remover.fit_transform)
X_train_without_stop_words.head()

4293                                            [g, w, r]
1978    [reply, win, 100, weekly, 2006, fifa, world, c...
3989    [hello, sort, town, already, dont, rush, home,...
3935                                [guoyang, n, u, told]
4078    [hey, sathya, till, dint, even, single, time, ...
Name: Message, dtype: object

### Étape 3 : **TF-IDF Vectorization**

In [103]:
X_train_encoded = pd.DataFrame(tfidf.fit_transform(X_train_without_stop_words.tolist()))
X_train_encoded.head()

,0,1,2,3,4,5,6,7,8,9,...,7542,7543,7544,7545,7546,7547,7548,7549,7550,7551
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Prétraitement des données de test

Appliquer le même pipeline au ensemble de test (sans réentraîner les transformateurs)

In [104]:
# Faire le prétraitement du test de la meme facon que le train


# Tokenizer -> StopWords -> TFIDFVectorizer


X_test_tokens = X_test['Message'].apply(tokenizer.tokenize)

X_test_without_stop_words = X_test_tokens.apply(stopwords_remover.transform)

X_test_encoded = pd.DataFrame(tfidf.transform(X_test_without_stop_words.tolist()))

print(f"X_test shape: {X_test_encoded.shape}")
print(f"X_test: {len(X_test_encoded)}")


X_test shape: (1114, 7552)
X_test: 1114


---

# 4. Encodage des Labels

Convertir les catégories texte en valeurs numériques

In [105]:
# ÉTAPE 2: Encodage  des catégories(y_train, y_test) 

le = CategoricalEncoder('label')

y_train = le.fit_transform(y_train)

In [106]:
y_test = le.transform(y_test)

---

# 5. Classification avec Régression Logistique

## Modèle 1 : Entraînement avec validation croisée

In [107]:
# ÉTAPE 3: Classification 
X_train = X_train_encoded


y_train_processed = y_train.squeeze() if isinstance(y_train, pd.DataFrame) else np.ravel(y_train)

y_test_processed = y_test.squeeze() if isinstance(y_test, pd.DataFrame) else np.ravel(y_test)

print("ÉTAPE 3: Entraînement du classifieur de spam avec la Regression Logistique")

model =  LogisticRegression() # logistic regression 

score = k_fold_cross_validation(model=model , X=X_train , y=y_train_processed , metric=f1_score , stratified=True)

print(f"Cross-validation - F1_Score global {score[0]}   ± {score[1]}")


print("\nRéentraînement du modèle sur l'ensemble complet...")
model.fit(X_train, y_train_processed)

ÉTAPE 3: Entraînement du classifieur de spam avec la Regression Logistique
Cross-validation - F1_Score global 0.0   ± 0.0

Réentraînement du modèle sur l'ensemble complet...


In [108]:
# Métriques

y_pred = model.predict(X_test_encoded)

print(f"Métriques finales sur l'ensemble de test\n")

# Accuracy (fonctionne pour multiclass)
print(f"Accuracy = {accuracy(y_test_processed , y_pred):.2f}")

# Macro-average: moyenne pour CHAQUE classe
all_classes = np.unique(y_test_processed)
precision_macro = np.mean([precision(y_test_processed, y_pred, positive_class=c) for c in all_classes])
recall_macro = np.mean([recall(y_test_processed, y_pred, positive_class=c) for c in all_classes])
f1_macro = np.mean([f1_score(y_test_processed, y_pred, positive_class=c) for c in all_classes])

print(f"Precision (macro) = {precision_macro:.2f}")
print(f"Recall (macro) = {recall_macro:.2f}")
print(f"F1_Score (macro) = {f1_macro:.2f}")

# Aussi pour la classe positive (supposée être 1)
print(f"\n--- Classe 1 (positive_class) ---")
print(f"Precision (class 1) = {precision(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"Recall (class 1) = {recall(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"F1_Score (class 1) = {f1_score(y_test_processed , y_pred, positive_class=1):.2f}")

Métriques finales sur l'ensemble de test

Accuracy = 0.87
Precision (macro) = 0.43
Recall (macro) = 0.50
F1_Score (macro) = 0.46

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.00
Recall (class 1) = 0.00
F1_Score (class 1) = 0.00


## Évaluation des métriques

In [89]:

y_pred = model.predict(X_test_encoded)
print(f"Métriques finales sur l'ensemble de test\n")

# Accuracy (fonctionne pour multiclass)
print(f"Accuracy = {accuracy(y_test_processed , y_pred):.2f}")

# Macro-average: moyenne pour CHAQUE classe
all_classes = np.unique(y_test_processed)
precision_macro = np.mean([precision(y_test_processed, y_pred, positive_class=c) for c in all_classes])
recall_macro = np.mean([recall(y_test_processed, y_pred, positive_class=c) for c in all_classes])
f1_macro = np.mean([f1_score(y_test_processed, y_pred, positive_class=c) for c in all_classes])

print(f"Precision (macro) = {precision_macro:.2f}")
print(f"Recall (macro) = {recall_macro:.2f}")
print(f"F1_Score (macro) = {f1_macro:.2f}")

# Aussi pour la classe positive (supposée être 1)
print(f"\n--- Classe 1 (positive_class) ---")
print(f"Precision (class 1) = {precision(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"Recall (class 1) = {recall(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"F1_Score (class 1) = {f1_score(y_test_processed , y_pred, positive_class=1):.2f}")

Métriques finales sur l'ensemble de test

Accuracy = 0.87
Precision (macro) = 0.43
Recall (macro) = 0.50
F1_Score (macro) = 0.46

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.00
Recall (class 1) = 0.00
F1_Score (class 1) = 0.00


A cause du déséquuilibre des classes le modele n'est pas performant pour la detection des mails classées comme spam

---

# 6. Optimisation : Gestion du Déséquilibre des Classes

## Recherche d'un meuilleur seuil

In [113]:

thresholds = np.arange(0.10, 0.21, 0.01)


probas = model.predict_proba(X_test_encoded)

for thresold in thresholds :
    y_pred = (probas >= 0.14).astype(np.int8)

    print("***"*20)
    print(f"Métriques  sur l'ensemble de test pour le seuil {thresold}\n")

    # Accuracy (fonctionne pour multiclass)
    print(f"Accuracy = {accuracy(y_test_processed , y_pred):.2f}")

    # Macro-average: moyenne pour CHAQUE classe
    all_classes = np.unique(y_test_processed)
    precision_macro = np.mean([precision(y_test_processed, y_pred, positive_class=c) for c in all_classes])
    recall_macro = np.mean([recall(y_test_processed, y_pred, positive_class=c) for c in all_classes])
    f1_macro = np.mean([f1_score(y_test_processed, y_pred, positive_class=c) for c in all_classes])

    print(f"Precision (macro) = {precision_macro:.2f}")
    print(f"Recall (macro) = {recall_macro:.2f}")
    print(f"F1_Score (macro) = {f1_macro:.2f}")

    # Aussi pour la classe positive (supposée être 1)
    print(f"\n--- Classe 1 (positive_class) ---")
    print(f"Precision (class 1) = {precision(y_test_processed , y_pred, positive_class=1):.2f}")
    print(f"Recall (class 1) = {recall(y_test_processed , y_pred, positive_class=1):.2f}")
    print(f"F1_Score (class 1) = {f1_score(y_test_processed , y_pred, positive_class=1):.2f}")

************************************************************
Métriques  sur l'ensemble de test pour le seuil 0.1

Accuracy = 0.95
Precision (macro) = 0.87
Recall (macro) = 0.95
F1_Score (macro) = 0.90

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.75
Recall (class 1) = 0.95
F1_Score (class 1) = 0.84
************************************************************
Métriques  sur l'ensemble de test pour le seuil 0.11

Accuracy = 0.95
Precision (macro) = 0.87
Recall (macro) = 0.95
F1_Score (macro) = 0.90

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.75
Recall (class 1) = 0.95
F1_Score (class 1) = 0.84
************************************************************
Métriques  sur l'ensemble de test pour le seuil 0.12

Accuracy = 0.95
Precision (macro) = 0.87
Recall (macro) = 0.95
F1_Score (macro) = 0.90

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.75
Recall (class 1) = 0.95
F1_Score (class 1) = 0.84
***********************************************************

Les métriques pour les classes **Ham** et **Spam** ne changent pas peu importe le seuil  
On peut donc garder un seuil de *_0.14_* pour notre model

In [114]:
probas = model.predict_proba(X_test_encoded)

y_pred = (probas >= 0.14).astype(np.int8)


print(f"Métriques finales sur l'ensemble de test\n")

# Accuracy (fonctionne pour multiclass)
print(f"Accuracy = {accuracy(y_test_processed , y_pred):.2f}")

# Macro-average: moyenne pour CHAQUE classe
all_classes = np.unique(y_test_processed)
precision_macro = np.mean([precision(y_test_processed, y_pred, positive_class=c) for c in all_classes])
recall_macro = np.mean([recall(y_test_processed, y_pred, positive_class=c) for c in all_classes])
f1_macro = np.mean([f1_score(y_test_processed, y_pred, positive_class=c) for c in all_classes])

print(f"Precision (macro) = {precision_macro:.2f}")
print(f"Recall (macro) = {recall_macro:.2f}")
print(f"F1_Score (macro) = {f1_macro:.2f}")

# Aussi pour la classe positive (supposée être 1)
print(f"\n--- Classe 1 (positive_class) ---")
print(f"Precision (class 1) = {precision(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"Recall (class 1) = {recall(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"F1_Score (class 1) = {f1_score(y_test_processed , y_pred, positive_class=1):.2f}")

Métriques finales sur l'ensemble de test

Accuracy = 0.95
Precision (macro) = 0.87
Recall (macro) = 0.95
F1_Score (macro) = 0.90

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.75
Recall (class 1) = 0.95
F1_Score (class 1) = 0.84


---

# 7. Prédictions sur du Texte Personnalisé

## Fonction pour prédire si un email est Spam ou Ham

In [146]:
def predict_email(email_text):
    """
    Prédire si un email est Spam ou Ham
    
    Args:
        email_text (str): Le texte de l'email à classifier
        
    Returns:
        str: "SPAM" ou "HAM"
    """
    # Pipeline NLP : Tokenization → StopWords → TF-IDF
    tokens = tokenizer.tokenize(email_text)
    clean_tokens = stopwords_remover.transform(tokens)
    X_new = pd.DataFrame(tfidf.transform([clean_tokens]))
    
    score_spam = model.predict_proba(X_new)[0]
    
    return { 
            'email_text' : email_text[:50] + '...' if len(email_text) > 50 else email_text , 
            'result' : "SPAM" if score_spam >= 0.14 else "HAM"
            }

## Exemples de prédictions

Testons le modèle sur plusieurs exemples d'emails pour voir comment il les classe

In [148]:
# Exemples d'emails pour tester

emails_test = [
    # Exemple 1: Email ham (légitime)
    "Hi John, How are you doing? Let me know about the project status. Thanks!",
    
    # Exemple 2: Email spam classique
    "CONGRATULATIONS! You have won 1 million dollars! Click here to claim your prize NOW!",
    
    # Exemple 3: Email ham (reunion)
    "Meeting rescheduled to tomorrow at 2 PM. Please confirm your attendance.",
    
    # Exemple 4: Email spam (offre douteuse)
    "FREE MONEY! Limited time offer. Earn $500 daily from home. No experience needed!",
    
    # Exemple 5: Email ham (notification)
    "Your order has been shipped. Tracking number: ABC123XYZ.",
]

print("="*80)
print("PRÉDICTIONS SUR DES EMAILS PERSONNALISÉS")
print("="*80 + "\n")

for i, email in enumerate(emails_test, 1):
    prediction = predict_email(email)
    print(f"Email {i}: {prediction['email_text']}")
    print(f"Prediction : {prediction['result']}")

PRÉDICTIONS SUR DES EMAILS PERSONNALISÉS

Email 1: Hi John, How are you doing? Let me know about the ...
Prediction : HAM
Email 2: CONGRATULATIONS! You have won 1 million dollars! C...
Prediction : SPAM
Email 3: Meeting rescheduled to tomorrow at 2 PM. Please co...
Prediction : SPAM
Email 4: FREE MONEY! Limited time offer. Earn $500 daily fr...
Prediction : SPAM
Email 5: Your order has been shipped. Tracking number: ABC1...
Prediction : HAM


## Testez votre propre email

Modifiez le texte ci-dessous pour tester le modèle sur vos propres emails

In [151]:
#  TESTEZ VOTRE PROPRE EMAIL ICI
custom_email = "Hello, check this amazing offer and try to win lot of prices!"

prediction = predict_email(custom_email)

print(f"Résultat: {prediction['result']}")

Résultat: SPAM


---

#  Résumé du Modèle et Performances

In [ ]:
# Résumé du modèle

print("="*80)
print("📊 RÉSUMÉ DU MODÈLE ET PERFORMANCES")
print("="*80 + "\n")

print("🔧 CONFIGURATION DU MODÈLE:")
print(f"  • Algorithme: Régression Logistique")
print(f"  • Vectorisation: TF-IDF (max 20000 features mais 7552 utilisées )")
print(f"  • Seuil de décision: 0.14 (optimisé pour détecter le spam)")
print(f"  • Validation: K-Fold Cross-Validation (stratifiée)")

print(f"\n📈 PERFORMANCES SUR L'ENSEMBLE DE TEST:")
print(f"  • Accuracy: {accuracy(y_test_processed, y_pred):.2%}")
print(f"  • Precision (macro): {precision_macro:.2%}")
print(f"  • Recall (macro): {recall_macro:.2%}")
print(f"  • F1-Score (macro): {f1_macro:.2%}")

print(f"\n🎯 POINTS CLÉS:")
print(f"  ✓ Gestion du déséquilibre des classes (Ham >> Spam)")
print(f"  ✓ Utilisation d'un seuil optimisé pour améliorer la détection du spam")
print(f"  ✓ Pipeline NLP complet: Tokenization → StopWords → TF-IDF")
print(f"  ✓ Fonction prédictive simple pour tester sur du texte personnalisé")

print(f"\n💡 CAS D'USAGE:")
print(f"  • Classifier automatiquement les emails entrants")
print(f"  • Filtrer les messages spam")
print(f"  • Améliorer l'expérience utilisateur en réduisant les faux positifs")

print("\n" + "="*80)

📊 RÉSUMÉ DU MODÈLE ET PERFORMANCES

🔧 CONFIGURATION DU MODÈLE:
  • Algorithme: Régression Logistique
  • Vectorisation: TF-IDF (max 20000 features)
  • Seuil de décision: 0.14 (optimisé pour détecter le spam)
  • Validation: K-Fold Cross-Validation (stratifiée)

📈 PERFORMANCES SUR L'ENSEMBLE DE TEST:
  • Accuracy: 95.06%
  • Precision (macro): 87.07%
  • Recall (macro): 94.88%
  • F1-Score (macro): 90.39%

🎯 POINTS CLÉS:
  ✓ Gestion du déséquilibre des classes (Ham >> Spam)
  ✓ Utilisation d'un seuil optimisé pour améliorer la détection du spam
  ✓ Pipeline NLP complet: Tokenization → StopWords → TF-IDF
  ✓ Fonction prédictive simple pour tester sur du texte personnalisé

💡 CAS D'USAGE:
  • Classifier automatiquement les emails entrants
  • Filtrer les messages spam
  • Améliorer l'expérience utilisateur en réduisant les faux positifs

